# Aegisub AI Non-Rigid Mesh Tracker (CoTracker3 for Google Colab)

Track complex **curved text, bending signs, waving cloth, crumpled/folding paper, and temporarily occluded planes** using Meta AI's **CoTracker3** transformer point tracker on a free Google Colab T4 GPU, then export `mesh_track.json` directly back into Aegisub.

### Before You Start
1. Enable GPU in Colab: **Runtime → Change runtime type → T4 GPU**.
2. In Aegisub, draw a 4-corner vector `\clip(m x1 y1 l x2 y2 x3 y3 x4 y4)` around the target sign on your **starting** (Forward) or **ending** (Backward) frame, and copy the clip tag.
3. **Tip:** For best speed and VRAM usage, trim your video to just the relevant few seconds (e.g., 20–250 frames) or specify `START_FRAME` / `END_FRAME` below.
4. Run all cells, download `mesh_track.json`, then in Aegisub choose **AI Mesh Import (Google Colab CoTracker3 JSON)**, click **Choose Mesh JSON...**, and check **Option A (Vector Drawing Mesh Warp)**, **Option B**, and/or **Option C**.

In [ ]:
#@title 1. Check GPU & Install Dependencies
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Switch to T4 GPU via Runtime -> Change runtime type for fast tracking.')

!pip -q install "imageio[ffmpeg]" opencv-python-headless einops timm tqdm

In [ ]:
#@title 2. Upload Video Clip (or Mount Google Drive)
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_VIDEO_PATH = "/content/drive/MyDrive/your_clip.mp4"  #@param {type:"string"}

UPLOADED_VIDEO_PATH = ""
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    UPLOADED_VIDEO_PATH = DRIVE_VIDEO_PATH
    print('Using Drive video:', UPLOADED_VIDEO_PATH)
else:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        UPLOADED_VIDEO_PATH = list(uploaded.keys())[0]
        print('Uploaded video:', UPLOADED_VIDEO_PATH)

In [ ]:
#@title 3. Tracking Configuration (Paste Aegisub \clip & Frame Range)
VIDEO_PATH = UPLOADED_VIDEO_PATH if UPLOADED_VIDEO_PATH else "clip.mp4"  #@param {type:"string"}
CLIP_STRING = r"\clip(m 400 200 l 900 200 900 500 400 500)"  #@param {type:"string"}
PLAY_RES_X = 1920  #@param {type:"number"}
PLAY_RES_Y = 1080  #@param {type:"number"}
START_FRAME = 0  #@param {type:"integer"}
END_FRAME = -1  #@param {type:"integer"}  # -1 = track to last frame of uploaded clip
TRACKING_DIRECTION = "forward"  #@param ["forward", "backward"]
GRID_COLS = 14  #@param {type:"slider", min:4, max:24, step:1}
GRID_ROWS = 8  #@param {type:"slider", min:4, max:24, step:1}
MAX_SHORT_SIDE = 720  #@param {type:"integer"}  # Downscale during AI inference if larger (0 = full res)
SMOOTHING_WINDOW = 0  #@param {type:"slider", min:0, max:10, step:1}
OUTPUT_JSON_PATH = "mesh_track.json"  #@param {type:"string"}

In [ ]:
#@title 4. Parse \clip, Load Frames & Preview Initial Mesh
import math
import os
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt

def parse_ass_clip_corners(clip_str: str) -> np.ndarray:
    m = re.search(r"\\i?clip\(([^)]*)\)", clip_str)
    payload = (m.group(1) if m else clip_str).strip()
    scale_denom = 1.0
    if "," in payload:
        parts = [p.strip() for p in payload.split(",")]
        if len(parts) == 2 and re.fullmatch(r"[-+]?\d+(?:\.\d+)?", parts[0]):
            scale_denom = 2.0 ** (float(parts[0]) - 1.0)
            payload = parts[1]
        else:
            raise ValueError("Rectangular \\clip is unsupported; paste a 4-point vector \\clip(m ... l ...).")
    tokens = re.findall(r"[a-zA-Z]+|[-+]?(?:\d+(?:\.\d*)?|\.\d+)", payload)
    coords = []
    for tok in tokens:
        if not tok.isalpha():
            coords.append(float(tok) / scale_denom)
    pts = [[coords[i], coords[i + 1]] for i in range(0, len(coords), 2)]
    if len(pts) == 5 and math.hypot(pts[4][0] - pts[0][0], pts[4][1] - pts[0][1]) <= 1.0:
        pts.pop()
    if len(pts) != 4:
        raise ValueError(f"Expected 4 \\clip corners, got {len(pts)}")
    arr = np.asarray(pts, dtype=np.float64)
    area2 = float(np.sum(arr[:, 0] * np.roll(arr[:, 1], -1) - np.roll(arr[:, 0], -1) * arr[:, 1]))
    if area2 < 0:
        arr = np.array([arr[0], arr[3], arr[2], arr[1]], dtype=np.float64)
    return arr

assert os.path.isfile(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
cap = cv2.VideoCapture(VIDEO_PATH)
vid_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
vid_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
start_f = max(0, int(START_FRAME))
end_f = (total_frames - 1) if END_FRAME < 0 else min(total_frames - 1, int(END_FRAME))
assert end_f >= start_f, f"Invalid frame range: {start_f}..{end_f}"

scale_infer = 1.0
short_side = min(vid_w, vid_h)
if MAX_SHORT_SIDE > 0 and short_side > MAX_SHORT_SIDE:
    scale_infer = MAX_SHORT_SIDE / float(short_side)
infer_w = max(64, (int(round(vid_w * scale_infer)) // 2) * 2)
infer_h = max(64, (int(round(vid_h * scale_infer)) // 2) * 2)

cap.set(cv2.CAP_PROP_POS_FRAMES, float(start_f))
frames_rgb = []
for _ in range(end_f - start_f + 1):
    ok, bgr = cap.read()
    if not ok or bgr is None:
        break
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if infer_w != vid_w or infer_h != vid_h:
        rgb = cv2.resize(rgb, (infer_w, infer_h), interpolation=cv2.INTER_AREA)
    frames_rgb.append(rgb)
cap.release()

num_frames = len(frames_rgb)
end_f = start_f + num_frames - 1
ref_rel = 0 if TRACKING_DIRECTION == "forward" else (num_frames - 1)
ref_f = start_f + ref_rel
print(f"Loaded {num_frames} frames ({start_f}..{end_f}) | video={vid_w}x{vid_h} | infer={infer_w}x{infer_h} | ref_frame={ref_f}")

res_x = float(PLAY_RES_X) if PLAY_RES_X > 0 else float(vid_w)
res_y = float(PLAY_RES_Y) if PLAY_RES_Y > 0 else float(vid_h)
quad_ass = parse_ass_clip_corners(CLIP_STRING)
quad_infer = quad_ass * np.array([infer_w / res_x, infer_h / res_y], dtype=np.float64)

unit_sq = np.array([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]], dtype=np.float32)
h_init = cv2.getPerspectiveTransform(unit_sq, quad_infer.astype(np.float32))
us = np.linspace(0.0, 1.0, int(GRID_COLS), dtype=np.float32)
vs = np.linspace(0.0, 1.0, int(GRID_ROWS), dtype=np.float32)
gx, gy = np.meshgrid(us, vs)
unit_pts = np.column_stack([gx.ravel(), gy.ravel()]).reshape(-1, 1, 2)
init_mesh_infer = cv2.perspectiveTransform(unit_pts, h_init).reshape(int(GRID_ROWS), int(GRID_COLS), 2)

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(frames_rgb[ref_rel])
for r in range(int(GRID_ROWS)):
    ax.plot(init_mesh_infer[r, :, 0], init_mesh_infer[r, :, 1], color='lime', linewidth=1.2)
for c in range(int(GRID_COLS)):
    ax.plot(init_mesh_infer[:, c, 0], init_mesh_infer[:, c, 1], color='lime', linewidth=1.2)
ax.scatter(quad_infer[:, 0], quad_infer[:, 1], color='yellow', s=45, zorder=5)
ax.set_title(f'Initial {GRID_COLS}x{GRID_ROWS} Mesh on Reference Frame {ref_f}')
ax.axis('off')
plt.show()

In [ ]:
#@title 5. Run CoTracker3 AI Point Mesh Tracking on GPU
import inspect
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
cotracker = torch.hub.load("facebookresearch/co-tracker", "cotracker3_offline").to(device)
cotracker.eval()

video_np = np.stack(frames_rgb, axis=0)  # (T, H, W, 3)
video_tensor = torch.from_numpy(video_np).permute(0, 3, 1, 2)[None].float().to(device)

flat_pts = init_mesh_infer.reshape(-1, 2)
t_col = np.full((len(flat_pts), 1), float(ref_rel), dtype=np.float32)
queries_np = np.hstack([t_col, flat_pts.astype(np.float32)])
queries_tensor = torch.from_numpy(queries_np)[None].to(device)

kwargs = {"queries": queries_tensor, "backward_tracking": (ref_rel > 0)}
if "add_support_grid" in inspect.signature(cotracker.forward).parameters:
    kwargs["add_support_grid"] = True

with torch.no_grad():
    pred_tracks, pred_visibility = cotracker(video_tensor, **kwargs)

tracks_infer = pred_tracks[0].detach().cpu().numpy().reshape(num_frames, int(GRID_ROWS), int(GRID_COLS), 2)
vis_arr = pred_visibility[0].detach().cpu().numpy()
if vis_arr.ndim == 3 and vis_arr.shape[-1] == 1:
    vis_arr = vis_arr[..., 0]
vis_arr = vis_arr.reshape(num_frames, int(GRID_ROWS), int(GRID_COLS)).astype(np.float64)

# Rescale tracked coordinates from inference resolution to exact ASS PlayRes
meshes_ass = tracks_infer.astype(np.float64) * np.array([res_x / infer_w, res_y / infer_h], dtype=np.float64)

# Pin exact user-supplied reference frame corners
meshes_ass[ref_rel] = init_mesh_infer.astype(np.float64) * np.array([res_x / infer_w, res_y / infer_h], dtype=np.float64)

if int(SMOOTHING_WINDOW) > 0 and num_frames > 2:
    radius = max(0, min(10, int(SMOOTHING_WINDOW)))
    smoothed = np.empty_like(meshes_ass)
    for t in range(num_frames):
        r_t = min(radius, t, num_frames - 1 - t)
        if t == ref_rel or r_t == 0:
            smoothed[t] = meshes_ass[t]
            continue
        sigma = max(0.5, r_t / 2.0)
        offsets = np.arange(-r_t, r_t + 1)
        w = np.exp(-0.5 * (offsets / sigma) ** 2)
        w /= np.sum(w)
        acc = np.zeros_like(meshes_ass[t])
        for k, wk in zip(offsets, w):
            acc += wk * meshes_ass[t + int(k)]
        smoothed[t] = acc
    meshes_ass = smoothed

print("CoTracker3 tracking complete! Mesh tensor shape:", meshes_ass.shape)

In [ ]:
#@title 6. Visual Verification (Sampled Frames + Optional Preview Video)
RENDER_PREVIEW_MP4 = False  #@param {type:"boolean"}

meshes_vis = meshes_ass * np.array([infer_w / res_x, infer_h / res_y], dtype=np.float64)
sample_indices = np.linspace(0, num_frames - 1, min(4, num_frames), dtype=int)
fig, axes = plt.subplots(1, len(sample_indices), figsize=(18, 4.5))
if len(sample_indices) == 1:
    axes = [axes]

for ax, idx in zip(axes, sample_indices):
    ax.imshow(frames_rgb[idx])
    m = meshes_vis[idx]
    v = vis_arr[idx]
    for r in range(int(GRID_ROWS)):
        ax.plot(m[r, :, 0], m[r, :, 1], color='cyan', linewidth=0.9)
    for c in range(int(GRID_COLS)):
        ax.plot(m[:, c, 0], m[:, c, 1], color='cyan', linewidth=0.9)
    vis_mask = v >= 0.5
    ax.scatter(m[vis_mask, 0], m[vis_mask, 1], color='lime', s=8)
    if np.any(~vis_mask):
        ax.scatter(m[~vis_mask, 0], m[~vis_mask, 1], color='red', s=12)
    ax.set_title(f'Frame {start_f + idx}')
    ax.axis('off')
plt.tight_layout()
plt.show()

if RENDER_PREVIEW_MP4:
    writer = cv2.VideoWriter('mesh_preview.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 24.0, (infer_w, infer_h))
    for idx in range(num_frames):
        bgr = cv2.cvtColor(frames_rgb[idx], cv2.COLOR_RGB2BGR)
        m = np.round(meshes_vis[idx]).astype(np.int32)
        for r in range(int(GRID_ROWS)):
            cv2.polylines(bgr, [m[r, :, :]], False, (255, 255, 0), 1, cv2.LINE_AA)
        for c in range(int(GRID_COLS)):
            cv2.polylines(bgr, [m[:, c, :]], False, (255, 255, 0), 1, cv2.LINE_AA)
        writer.write(bgr)
    writer.release()
    print('Saved mesh_preview.mp4')

In [ ]:
#@title 7. Export Mesh JSON & Download for Aegisub
import json

export_payload = {
    "format": "aegisub_perspective_mesh_v1",
    "start_frame": int(start_f),
    "end_frame": int(end_f),
    "ref_frame": int(ref_f),
    "play_res_x": float(res_x),
    "play_res_y": float(res_y),
    "grid_cols": int(GRID_COLS),
    "grid_rows": int(GRID_ROWS),
    "frames": [
        {
            "frame": int(start_f + t),
            "points": np.round(meshes_ass[t], 3).tolist(),
            "visibility": np.round(vis_arr[t], 4).tolist(),
        }
        for t in range(num_frames)
    ],
}

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(export_payload, f)

print(f"Exported {num_frames} frames ({GRID_COLS}x{GRID_ROWS} mesh) to {OUTPUT_JSON_PATH}")
try:
    from google.colab import files
    files.download(OUTPUT_JSON_PATH)
except ImportError:
    pass